In [ ]:
%pip install -q polars plotly gdown python-dotenv kailash-ml


# ════════════════════════════════════════════════════════════════════════
# ASCENT07 — Exercise 7: CNNs for Image Classification
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build a CNN with convolution, pooling, dropout layers for
#   image classification, then export to ONNX via OnnxBridge.
#
# TASKS:
#   1. Implement convolution operation from scratch
#   2. Build CNN architecture (conv → pool → conv → pool → fc)
#   3. Train with dropout regularization
#   4. Evaluate and visualize filters with ModelVisualizer
#   5. Export to ONNX via OnnxBridge and validate
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import random

import polars as pl

from kailash_ml import ModelVisualizer, OnnxBridge

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



## TASK 1: Implement convolution operation from scratch


In [ ]:
loader = ASCENTDataLoader()
data = loader.load("ascent07", "fashion_mnist_sample.parquet")

pixel_cols = [c for c in data.columns if c != "label"]
n_samples = data.height
print(f"=== Fashion-MNIST: {n_samples} samples, 28×28 images ===")


def to_image(row_values: list[float], h: int = 28, w: int = 28) -> list[list[float]]:
    """Convert flat pixel list to 2D image."""
    return [row_values[i * w : (i + 1) * w] for i in range(h)]


def conv2d(
    image: list[list[float]],
    kernel: list[list[float]],
    stride: int = 1,
    padding: int = 0,
) -> list[list[float]]:
    """2D convolution: slide kernel over image, compute dot product at each position."""
    h, w = len(image), len(image[0])
    kh, kw = len(kernel), len(kernel[0])

    if padding > 0:
        padded = [[0.0] * (w + 2 * padding) for _ in range(h + 2 * padding)]
        for i in range(h):
            for j in range(w):
                padded[i + padding][j + padding] = image[i][j]
        image = padded
        h, w = len(image), len(image[0])

    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    output = [[0.0] * out_w for _ in range(out_h)]

    # TODO: Slide kernel over each output position (i, j).
    # Accumulate image[i*stride+ki][j*stride+kj] * kernel[ki][kj] over (ki, kj).
    # Write the dot-product sum to output[i][j].
    for i in range(out_h):
        for j in range(out_w):
            ____
            for ki in range(kh):
                for kj in range(kw):
                    ____
            ____

    return output


sample_pixels = data.select(pixel_cols).row(0)
sample_img = to_image([v / 255.0 for v in sample_pixels])

horizontal_kernel = [[-1, -1, -1], [0, 0, 0], [1, 1, 1]]
vertical_kernel = [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]

h_edges = conv2d(sample_img, horizontal_kernel, padding=1)
v_edges = conv2d(sample_img, vertical_kernel, padding=1)

print(f"Input image shape: 28×28")
print(f"Kernel shape: 3×3")
print(f"Output shape (padding=1): {len(h_edges)}×{len(h_edges[0])}")
print(f"Edge detector reveals structure that a fully-connected layer cannot see.")



## TASK 2: Build CNN architecture


In [ ]:
def max_pool2d(feature_map: list[list[float]], pool_size: int = 2) -> list[list[float]]:
    """Max pooling: downsample by taking max in each pool window."""
    h, w = len(feature_map), len(feature_map[0])
    out_h, out_w = h // pool_size, w // pool_size
    output = [[0.0] * out_w for _ in range(out_h)]
    for i in range(out_h):
        for j in range(out_w):
            vals = []
            for pi in range(pool_size):
                for pj in range(pool_size):
                    vals.append(feature_map[i * pool_size + pi][j * pool_size + pj])
            output[i][j] = max(vals)
    return output


def relu_2d(feature_map: list[list[float]]) -> list[list[float]]:
    return [[max(0.0, v) for v in row] for row in feature_map]


def flatten(feature_maps: list[list[list[float]]]) -> list[float]:
    """Flatten list of 2D feature maps into 1D vector."""
    result = []
    for fm in feature_maps:
        for row in fm:
            result.extend(row)
    return result


class SimpleCNN:
    """Minimal CNN: conv(3x3,4 filters)→pool→conv(3x3,8 filters)→pool→fc→softmax."""

    def __init__(self, n_classes: int = 10):
        random.seed(42)
        self.n_classes = n_classes
        # TODO: Initialize conv1_filters (4 × 3×3, gauss(0,0.3)), conv1_bias (4 zeros),
        # conv2_filters (8 × 3×3, gauss(0,0.3)), conv2_bias (8 zeros),
        # fc_w (8*7*7 × n_classes, gauss(0,0.01)), fc_b (n_classes zeros).
        ____
        ____
        ____
        ____
        ____
        ____

    def forward(self, image: list[list[float]]) -> list[float]:
        """Forward pass: conv1→relu→pool → conv2→relu→pool → flatten→fc→softmax."""
        # TODO: Conv1 block — for each of 4 filters: conv2d(padding=1) → relu_2d → max_pool2d.
        # Conv2 block — for each of 8 filters: same; input = conv1_out[f_idx % 4].
        # Flatten conv2_out, compute logits via fc_w/fc_b, apply stable softmax.
        ____
        ____
        ____
        ____
        ____
        ____
        ____
        ____
        ____
        ____
        ____


cnn = SimpleCNN(n_classes=10)
sample_probs = cnn.forward(sample_img)
print(f"Sample prediction: class {sample_probs.index(max(sample_probs))}")



## TASK 3: Train with dropout


In [ ]:
def dropout(
    values: list[float], rate: float = 0.5, training: bool = True
) -> list[float]:
    """Inverted dropout: zero out `rate` fraction of units, scale survivors by 1/(1-rate)."""
    if not training:
        return values
    # TODO: Build mask (0.0 if random()<rate else 1/(1-rate)); return values * mask element-wise.
    ____
    ____


print(
    f"Dropout rate: 0.5 — zeros 50% of activations; scaled by 1/(1-rate) so test needs no change."
)

n_train_cnn = int(data.height * 0.8)
train_cnn = data[:n_train_cnn]
test_cnn = data[n_train_cnn:]

# TODO: Forward pass each of the first 20 training images; compute CE loss; track train_losses.
train_losses = []
____
____
____
____
____
print(f"Training complete — final loss: {train_losses[-1]:.4f}")



## TASK 4: Visualize filters with ModelVisualizer


In [ ]:
viz = ModelVisualizer()

# TODO: For each conv1 filter, compute flat_vals and print min/max.
for i, filt in enumerate(cnn.conv1_filters):
    ____
    ____



## TASK 5: Export to ONNX via OnnxBridge


In [ ]:
print(f"\n=== ONNX Export ===")
print(f"OnnxBridge.export(model, framework, output_path=...) converts to ONNX.")
print(f"Example: bridge.export(model, 'sklearn', output_path='model.onnx')")
print(f"Skipping actual export (hand-built CNN is not sklearn/PyTorch compatible).")
print(f"ONNX model is portable: runs on any ONNX runtime (C++, JS, mobile)")

print("\n✓ Exercise 7 complete — CNN from scratch + ONNX export via OnnxBridge")

